1. Set Up & Installation

In [31]:
import torch
import torch.nn as nn
from transformers import RTDetrForObjectDetection
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.preprocess import quant_pre_process

import os
import numpy as np
import onnxruntime
from onnxruntime.quantization import CalibrationDataReader
import cv2
from PIL import Image

2. Parameter Config

In [32]:
CLASS_MAP = {
    0: "Excavator", 
    1: "Pillar", 
    2: "Rock", 
    3: "Traffic Cone", 
    4: "Truck"
}
label2id = {v: k for k, v in CLASS_MAP.items()}

hf_model = RTDetrForObjectDetection.from_pretrained(
    "PekingU/rtdetr_r18vd",  
    id2label=CLASS_MAP, 
    label2id=label2id,
    ignore_mismatched_sizes=True
)

Loading weights: 100%|██████████| 526/526 [00:00<00:00, 1157.99it/s, Materializing param=model.encoder_input_proj.2.1.weight]                                                
RTDetrForObjectDetection LOAD REPORT from: PekingU/rtdetr_r18vd
Key                                        | Status   |                                                                                        
-------------------------------------------+----------+----------------------------------------------------------------------------------------
model.decoder.class_embed.{0, 1, 2}.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80]) vs model:torch.Size([5])          
model.decoder.class_embed.{0, 1, 2}.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([80, 256]) vs model:torch.Size([5, 256])
model.denoising_class_embed.weight         | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81, 256]) vs model:torch.Size([6, 256])
model.enc_score_head.bias                 

3. Define Wrapper & Model

In [33]:
print("Input weights from Hugging Face model: best_rt_detr_model.pth")
model_path = r"D:\Skripsi_Raphaela\rt_detr\program\best_rt_detr_model.pth"
checkpoint = torch.load(model_path, map_location='cpu')

# Anticipating if file is saved as dictionary
if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    hf_model.load_state_dict(checkpoint["state_dict"])
elif isinstance(checkpoint, dict):
    hf_model.load_state_dict(checkpoint)
else:
    hf_model = checkpoint

hf_model.eval()

# Create wrapper so model output becomes Tensor (logits and pred_boxes) instead of a Dictionary (custom object)
class RTDETR_ONNX_Wrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        out = self.model(pixel_values=pixel_values)
        # Take 2 variables for Raspi: logits and pred_boxes
        return out.logits, out.pred_boxes

wrapped_model = RTDETR_ONNX_Wrapper(hf_model)

Input weights from Hugging Face model: best_rt_detr_model.pth


C:\Users\pengguna\AppData\Local\Temp\ipykernel_23696\283396803.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location='cpu')


4. Export to ONNX (FP32)

In [34]:
print("Converting to ONNX...")
onnx_model_path = r"D:\Skripsi_Raphaela\rt_detr\program\rt_detr_fp32.onnx"

# IMG_SIZE = (1024, 1024)
# Format: (Batch_size, Channels, Height, Width)
dummy_input = torch.randn(1, 3, 1024, 1024)

torch.onnx.export(
    wrapped_model,
    dummy_input,
    onnx_model_path,
    export_params=True,
    opset_version=16,
    do_constant_folding=True,
    input_names=['pixel_values'],
    output_names=['logits', 'pred_boxes'],
    dynamic_axes={
        'pixel_values': {0: 'batch_size'},
        'logits': {0: 'batch_size'},
        'pred_boxes': {0: 'batch_size'}
    }
)
print(f"ONNX FP32 saved in : {onnx_model_path}")

Converting to ONNX...


d:\Skripsi_Raphaela\rt_detr\program\.venv\Lib\site-packages\transformers\models\rt_detr\modeling_rt_detr_resnet.py:92: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_channels != self.num_channels:
d:\Skripsi_Raphaela\rt_detr\program\.venv\Lib\site-packages\transformers\integrations\sdpa_attention.py:77: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  is_causal = query.shape[2] > 1 and attention_mask is None and is_causal
d:\Skripsi_Raphaela\rt_detr\program\.venv\Lib\site-packages\transformers\models\rt_detr\modeling_rt_detr.py:1599: TracerWarning: 

ONNX FP32 saved in : D:\Skripsi_Raphaela\rt_detr\program\rt_detr_fp32.onnx


5. Preprocess ONNX

In [35]:
print("Pre-processing ONNX...")
preprocessed_model_path = r"D:\Skripsi_Raphaela\rt_detr\program\rt_detr_fp32_prep.onnx"

quant_pre_process(
    input_model_path=onnx_model_path, 
    output_model_path=preprocessed_model_path, 
    skip_optimization=False,
    skip_symbolic_shape=True
)

print(f"Preprocessed ONNX FP32 saved in : {preprocessed_model_path}")


Pre-processing ONNX...
Preprocessed ONNX FP32 saved in : D:\Skripsi_Raphaela\rt_detr\program\rt_detr_fp32_prep.onnx


6. Static Quantization

In [36]:
class RTDetrCalibrationDataReader(CalibrationDataReader):
    def __init__(self, image_folder, input_name='pixel_values', img_size=1024):
        self.image_folder = image_folder
        self.input_name = input_name
        self.img_size = img_size
        self.image_list = [os.path.join(image_folder, f) for f in os.listdir(image_folder) 
                          if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        self.image_list = self.image_list[:100]  # Gunakan 100 gambar untuk kalibrasi
        self.preprocess_flag = True
        self.enum_data = None

    def get_next(self):
        if self.enum_data is None:
            self.enum_data = iter(self.image_list)
        
        image_path = next(self.enum_data, None)
        if image_path is None:
            return None

        # Pre-process image according to training (resize, normalize, dll)
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        
        # Normalize to [0, 1] (same as training)
        img = img.astype(np.float32) / 255.0
        
        # Mean & Std ImageNet
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = (img - mean) / std
        
        # HWC to CHW and add Batch Dimension
        img = img.transpose(2, 0, 1)
        img = np.expand_dims(img, axis=0)
        img = img.astype(np.float32)
        
        return {self.input_name: img}

In [37]:
PREP_ONNX_PATH = preprocessed_model_path # Taking the output from Step 5
INT8_ONNX_PATH = r"D:\Skripsi_Raphaela\rt_detr\program\rt_detr_int8.onnx"

print("Starting Dynamic Quantization (INT8) for RT-DETR...")
print("Note: This will quantize the model weights to INT8 to reduce size, ")
print("but keep the activations in FP32 during inference to preserve Transformer stability.\n")

# Execute Dynamic Quantization
quantize_dynamic(
    model_input=PREP_ONNX_PATH,
    model_output=INT8_ONNX_PATH,
    weight_type=QuantType.QInt8
)

print(f"Success! Dynamic INT8 model saved at: {INT8_ONNX_PATH}")

Starting Dynamic Quantization (INT8) for RT-DETR...
Note: This will quantize the model weights to INT8 to reduce size, 
but keep the activations in FP32 during inference to preserve Transformer stability.

Success! Dynamic INT8 model saved at: D:\Skripsi_Raphaela\rt_detr\program\rt_detr_int8.onnx
